# 3주차 — 개발 환경과 LangChain 첫 체인 (Colab판)

「최신인공지능」 2026 · 3주차 실습 · 2026년 9월 11일 (금)

| 실습 | 교시 | 내용 |
|------|------|------|
| 실습 1 | 1교시 | 키를 코드에서 분리하기 (Colab 시크릿) |
| 실습 2 | 2교시 | LangChain 첫 호출 — `ChatOllama` |
| 실습 3 ★ | 2교시 | `ollama.chat()` vs `ChatOllama` |
| 실습 4 | 2교시 | 메시지 타입 (System / Human / AI) |
| — | 3교시 | `ChatPromptTemplate` · `StrOutputParser` |
| 실습 5 ★ | 3교시 | **`prompt \| llm \| parser` 체인 조립** |

> ### ⚠️ 이 노트북으로 대체되지 **않는** 부분이 있습니다
>
> 1교시의 **가상환경(`python -m venv`) · VS Code 인터프리터 지정 · `.gitignore`** 는
> Colab에 대응물이 없습니다. **실습실 PC에서 반드시 따로 진행**하십시오.
> 이 노트북은 그중 **"키를 코드에서 분리한다"** 는 원칙만 Colab 방식(시크릿)으로 옮긴 것입니다.
>
> | 실습실 (원본) | Colab (이 노트북) |
> |---|---|
> | `.env` 파일 + `python-dotenv` | 좌측 🔑 **보안 비밀** 패널 |
> | `.gitignore` 로 커밋 차단 | 애초에 저장소에 안 들어감 |
> | `os.getenv("이름")` | **`os.getenv("이름")` — 똑같습니다** ★ |
>
> 코드에서 **이름으로만 참조한다**는 점은 양쪽이 완전히 같습니다. 그것이 이 실습의 요점입니다.

## 0. 환경 준비

아래 셀을 **한 번** 실행합니다. 런타임이 끊기면 다시 실행하십시오 (여러 번 실행해도 안전합니다).

- 모델은 매 세션 새로 내려받습니다 — **처음 실행은 3~5분** 걸립니다.
- GPU가 없으면 자동으로 작은 모델(`gemma3:1b`)로 내려갑니다.
  상단 메뉴 **[런타임] > [런타임 유형 변경] > T4 GPU** 로 먼저 바꿔 두는 편이 좋습니다.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Colab 환경 준비 — 매 세션 1회 실행 (재실행 안전)
# ══════════════════════════════════════════════════════════════
WEEK_MODELS   = ["chat"]                      # chat · small · embed · tool
WEEK_PACKAGES = "langchain langchain-core langchain-ollama python-dotenv ollama"
WEEK_SECRETS  = ["OPENAI_API_KEY"]            # 없어도 3주차 실습은 전부 진행됩니다

# ──────────────────────────────────────────────────────────────
import os, shutil, subprocess, sys, time, urllib.request

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

# 1) GPU 유무 → 대화 모델 결정
GPU   = shutil.which("nvidia-smi") is not None and sh("nvidia-smi").returncode == 0
CHAT  = os.environ.setdefault("MODEL",       "gemma3:4b" if GPU else "gemma3:1b")
SMALL = os.environ.setdefault("SMALL_MODEL", "gemma3:1b")
EMBED = os.environ.setdefault("EMBED_MODEL", "nomic-embed-text")
TOOL  = os.environ.setdefault("TOOL_MODEL",  "qwen3:4b")
PICK  = {"chat": CHAT, "small": SMALL, "embed": EMBED, "tool": TOOL}

print(f"[1/5] 런타임   {'GPU 있음 ✅' if GPU else 'CPU 전용 ⚠️'}   →  대화 모델 {CHAT}")
if not GPU:
    print("       [런타임] > [런타임 유형 변경] > T4 GPU 로 바꾸면 4b 모델을 쓸 수 있습니다.")

# 2) 패키지
print("[2/5] 패키지 설치 중…")
r = sh(f"{sys.executable} -m pip install -q {WEEK_PACKAGES}")
print("       ✅ 완료" if r.returncode == 0 else "       ❌ 실패\n" + r.stderr[-600:])

# 3) Ollama 설치
if shutil.which("ollama") is None:
    print("[3/5] Ollama 설치 중… (약 30초)")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
print("[3/5] Ollama  " + ("✅ 준비됨" if shutil.which("ollama") else "❌ 설치 실패"))

# 4) Ollama 서버 기동
def alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not alive():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if alive():
            break
        time.sleep(1)
print("[4/5] 서버    " + ("✅ 응답함" if alive() else "❌ 미응답 — 이 셀을 다시 실행하세요"))

# 5) 모델 내려받기
have = {ln.split()[0] for ln in sh("ollama list").stdout.splitlines()[1:] if ln.strip()}
for key in WEEK_MODELS:
    name = PICK[key]
    if name in have:
        print(f"[5/5] {name:<20s} ✅ 이미 있음")
        continue
    print(f"[5/5] {name:<20s} ⏳ 내려받는 중… (진행 표시 없이 수 분 걸립니다)")
    t0 = time.time()
    r = sh(f"ollama pull {name}")
    print(f"       {'✅ 완료' if r.returncode == 0 else '❌ 실패'}  ({time.time() - t0:.0f}초)")
    if r.returncode != 0:
        print(r.stderr[-400:])

# 6) 시크릿 (Colab 좌측 🔑 패널에서 등록 + '노트북 액세스' 켜기)
for k in WEEK_SECRETS:
    if not os.getenv(k) and IN_COLAB:
        try:
            from google.colab import userdata
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
    print(f"[키]  {k:<20s} " + ("✅ 설정됨" if os.getenv(k) else "⬜ 없음 (없어도 진행됩니다)"))

print("\n" + "=" * 62)
print(f"준비 완료 — 이 노트북의 MODEL 은 '{CHAT}' 입니다")
print("=" * 62)

## 실습 1 (1교시) — 키는 코드가 아니라 바깥에 둔다

**핵심 원칙**

- 키는 **코드에 적지 않는다**
- 코드에서는 **이름으로만** 참조한다
- 값 자체는 **절대 화면에 출력하지 않는다** ★

Colab에서는 좌측 사이드바의 🔑 **보안 비밀(Secrets)** 에 등록하고
해당 키의 **"노트북 액세스"** 토글을 켜면 됩니다. (실습실에서는 이 자리가 `.env` 파일입니다)

In [ ]:
import os

# 이번 학기에 쓸 키 목록
KEYS = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY"]


def mask(value: str) -> str:
    """키가 맞는지 눈으로만 확인할 수 있게 가린다.

    화면 공유·스크린샷·발표 자료로 유출되지 않도록
    앞 4글자 외에는 모두 * 로 덮는다.
    """
    if len(value) <= 4:
        return "*" * len(value)
    return value[:4] + "*" * (len(value) - 4)


print("=" * 50)
print("  키 로드 확인")
print("=" * 50)

for name in KEYS:
    value = os.getenv(name)

    # ❌ print(value)          ← 절대 금지. 키 전체가 그대로 노출된다
    # ✅ 존재 여부만 확인한다
    if value:
        print(f"  [O] {name:20s} 로드됨  ({mask(value)}, {len(value)}자)")
    else:
        print(f"  [X] {name:20s} 없음")

print("-" * 50)
print("  3주차는 로컬 Ollama만 쓰므로 [X] 여도 정상입니다.")
print("  키는 4주차 수업 중에 배포합니다.")
print("=" * 50)

> ### ⚠️ 실습실에서 반드시 따로 할 것
>
> Colab에는 `git`으로 관리하는 작업 폴더가 없습니다. 아래는 **실습실 PC 터미널에서** 확인하십시오.
>
> ```bash
> git status
> git check-ignore -v .env      # .gitignore:N:.env  이 나오면 정상
> ```
>
> `.env` 를 커밋하는 사고는 **공용 키가 통째로 노출**되어 전체 실습이 중단되는 사고입니다.
> 과제 1의 채점 항목이기도 합니다.

## 실습 2 (2교시) — LangChain 첫 호출 `ChatOllama`

작년에는 `ollama.chat()` 을 직접 불렀습니다. 이번에는 LangChain 규격으로 감싼 `ChatOllama` 를 씁니다.

**관찰 포인트**

1. 반환 타입이 문자열이 아니라 `AIMessage` **객체**다
2. 텍스트는 `.content` 안에 있다
3. `invoke()` 는 LangChain 모든 부품이 공유하는 실행 메서드다 ★

In [ ]:
import os
from langchain_ollama import ChatOllama

MODEL = os.environ["MODEL"]        # 부트스트랩이 정해 준 모델 (gemma3:4b 또는 1b)

# 1) 모델 객체 생성
#    temperature: 0에 가까울수록 일관된 답, 1에 가까울수록 다양한 답
llm = ChatOllama(model=MODEL, temperature=0.7)

# 2) 호출 — 문자열을 그대로 넘길 수도 있다
response = llm.invoke("파이썬의 장점 3가지를 각각 한 문장으로 알려줘.")

# 3) 결과 확인
print("반환 타입:", type(response))
#    → <class 'langchain_core.messages.ai.AIMessage'>
#      문자열이 아니라 객체로 돌아온다는 점에 주목
print()
print("── 응답 내용 ──────────────────────────────")
print(response.content)            # 실제 텍스트는 .content 안에 있다
print("──────────────────────────────────────────")

In [ ]:
# ── 응답 객체 들여다보기 ──────────────────────
# 왜 객체로 돌려주나?
#   텍스트만 주면 토큰 사용량·모델명·종료 이유를 알 수 없다.
#   6주차 LangSmith 에서 이 메타데이터가 비용·성능 분석의 원천이 된다.
print("메타데이터:", response.response_metadata)
print()
print("사용 토큰 :", response.usage_metadata)

## 실습 3 ★ (2교시) — `ollama.chat()` vs `ChatOllama`

이번 주차의 핵심 실습입니다.

- **방식 A** — 작년 방식 : 공급자 전용 SDK (`ollama`)
- **방식 B** — 이번 학기 : LangChain (`ChatOllama`)

결과는 같습니다. 코드는 다릅니다. → **무엇이 좋아졌나? 그리고 무엇이 감춰졌나?**

In [ ]:
import os

MODEL    = os.environ["MODEL"]
SYSTEM   = "당신은 한 문장으로만 답하는 비서입니다."
QUESTION = "대한민국의 수도는?"


# ══════════════════════════════════════════════════
# 방식 A — 작년 방식 : 공급자 전용 SDK
# ══════════════════════════════════════════════════
import ollama

resp = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": QUESTION},
    ],
)

print("[A] ollama.chat()")
print("   결과:", resp["message"]["content"].strip())
print("   타입:", type(resp))
# 결과를 꺼내려면 응답 구조를 외워야 한다:  resp["message"]["content"]
# 키 이름을 틀리면 실행 후에야 KeyError 로 알게 된다.
#
# 🔶 최신 ollama 패키지는 순수 dict 가 아니라 ollama._types.ChatResponse 를
#    돌려줍니다(딕셔너리처럼 [] 접근은 그대로 됨). 화면에 찍히는 타입이
#    강의안의 'dict' 와 달라도 핵심 논지는 그대로입니다 —
#    "공급자마다 응답 구조가 제각각이고, 그 구조를 외워야 한다".

In [ ]:
# ══════════════════════════════════════════════════
# 방식 B — 이번 학기 방식 : LangChain
# ══════════════════════════════════════════════════
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODEL)
resp_b = llm.invoke(
    [
        SystemMessage(content=SYSTEM),
        HumanMessage(content=QUESTION),
    ]
)

print("[B] ChatOllama.invoke()")
print("   결과:", resp_b.content.strip())
print("   타입:", type(resp_b))
# 결과 접근은 항상 .content — 공급자가 바뀌어도 동일하다.

### 관찰 — 무엇이 같고 무엇이 다른가

| 항목 | A: `ollama.chat()` | B: `ChatOllama` |
|---|---|---|
| 결과 텍스트 | 같음 | 같음 |
| 결과 접근 | `resp["message"]["content"]` | `resp.content` |
| 반환 타입 | dict / ChatResponse (버전마다 다름) | `AIMessage` |
| 메시지 표현 | 딕셔너리 | 타입 객체 |
| 오타 검출 | 실행 후 `KeyError` | 에디터가 미리 잡음 |

여기까지는 큰 차이가 없어 보입니다. **오히려 A 가 짧습니다.**
진짜 차이는 아래 질문에서 드러납니다.

### 4-2. 결정적 질문 — "OpenAI 로 바꾸려면?" ★

**방식 A — 전부 다시 쓴다**

```python
import ollama
resp = ollama.chat(model="gemma3:4b", messages=[...])
text = resp["message"]["content"]

# ↓ OpenAI 로 바꾸면 — 라이브러리·함수·응답 구조가 전부 다름
from openai import OpenAI
client = OpenAI()
resp = client.chat.completions.create(model="gpt-...", messages=[...])
text = resp.choices[0].message.content     # 접근 경로도 다르다!
```

**방식 B — 한 줄이다**

```python
llm = ChatOllama(model="gemma3:4b")
# ↓ 이 줄만 바꾼다
llm = ChatOpenAI(model="gpt-...")

# 아래는 그대로
resp = llm.invoke([SystemMessage(...), HumanMessage(...)])
text = resp.content
```

→ **4주차**에서 로컬 ↔ OpenAI 교체를 직접 해 봅니다.

### 4-3. 균형 — 무엇이 감춰지는가 ⚖️

| 얻는 것 | 대가 |
|---|---|
| 모델 교체가 한 줄 | 실제 HTTP 요청이 어떻게 나가는지 안 보인다 |
| 공통 인터페이스 | 공급자 고유 기능은 못 쓰거나 우회해야 함 |
| 검증된 부품 | 버전 변화가 빨라 따라가야 한다 |
| 생태계·자료 | 간단한 작업엔 과한 의존성 |

| 상황 | A (전용 SDK) | B (LangChain) |
|---|---|---|
| 모델 하나만 쓰는 간단한 스크립트 | ✅ 충분 | 과함 |
| 모델을 바꿔가며 비교해야 함 | 힘듦 | ✅ |
| RAG·에이전트처럼 부품이 많음 | 매우 힘듦 | ✅ |
| 공급자 최신 기능을 바로 써야 함 | ✅ | 지원 대기 |

> **결론**: LangChain 은 은탄환이 아닙니다.
> 다만 이번 학기에 만들 것 — 검색·도구·상태가 얽힌 애플리케이션 — 에서는
> 직접 짜는 비용이 훨씬 큽니다. 그래서 씁니다.

## 실습 4 (2교시) — 메시지 타입 System / Human / AI

선행 과목에서는 `{"role": "system", "content": ...}` 딕셔너리를 썼습니다.
LangChain 은 이를 **타입이 있는 객체**로 다룹니다.

| LangChain | 딕셔너리 | 역할 |
|---|---|---|
| `SystemMessage` | `{"role": "system"}` | AI의 역할·규칙 설정 |
| `HumanMessage` | `{"role": "user"}` | 사용자 입력 |
| `AIMessage` | `{"role": "assistant"}` | AI의 응답 |

딕셔너리 대비 장점 — 오타(`"sytem"`)가 나면 실행 전에 잡히고, 에디터가 자동완성해 줍니다.

In [ ]:
import os
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

llm = ChatOllama(model=os.environ["MODEL"])

# ── 1턴 ────────────────────────────────────────
messages = [
    SystemMessage(content="당신은 친절한 파이썬 튜터입니다. 초보자 눈높이로 답하세요."),
    HumanMessage(content="리스트와 튜플의 차이가 뭐야?"),
]

response = llm.invoke(messages)
print("[1턴] 리스트와 튜플의 차이가 뭐야?")
print(response.content)

In [ ]:
# ── 2턴: 대화 이력 이어가기 ────────────────────
# 모델의 답(AIMessage)을 그대로 리스트에 넣고, 다음 질문을 덧붙인다.
messages.append(response)                              # AIMessage 를 변환 없이 append
messages.append(HumanMessage(content="그럼 언제 튜플을 써?"))

response2 = llm.invoke(messages)
print("[2턴] 그럼 언제 튜플을 써?")
print(response2.content)
print("=" * 50)

# 지금 대화 이력이 몇 개인지 확인
messages.append(response2)
print(f"누적 메시지 {len(messages)}개")
for i, m in enumerate(messages, start=1):
    kind = type(m).__name__
    preview = m.content.replace("\n", " ")[:30]
    print(f"  {i}. {kind:15s} {preview}...")

> ### ⚠️ 여기서 멈춰서 생각해 볼 것
>
> "2주차의 ② **기억 못 한다**" 가 해결됐나?
>
> → **아닙니다.** 여전히 우리가 손으로 리스트를 관리하고 있습니다.
> 매 호출마다 전체 이력을 다시 보냅니다(= 토큰을 다시 냅니다).
> 진짜 해결은 **13주차 체크포인터**에서 이뤄집니다.

In [ ]:
# AIMessage 를 직접 만들어 이력에 끼워 넣는 예시.
# (이력을 파일에서 복원할 때 이렇게 쓴다)
restored = [
    SystemMessage(content="당신은 친절한 파이썬 튜터입니다."),
    HumanMessage(content="리스트와 튜플의 차이가 뭐야?"),
    AIMessage(content="리스트는 수정 가능하고, 튜플은 수정 불가능합니다."),
]
print(f"손으로 복원한 이력: {len(restored)}개")

## 3교시 1절 — `ChatPromptTemplate`: 프롬프트를 부품으로

f-string 으로 충분해 보이는데 무엇이 문제인가?

```python
topic = "파이썬"
prompt = f"{level}에게 {topic}의 장점 3가지를 설명하세요."
```

| 문제 | 내용 |
|---|---|
| 재사용 불가 | 다른 파일에서 쓰려면 복사해야 함 |
| 변수 누락 미검출 | `level` 을 안 넘기면 `NameError` 가 런타임에 터짐 |
| 버전 관리 불가 | 무엇이 바뀌었는지 추적 어려움 |
| **체인에 못 끼움** ★ | f-string 은 문자열일 뿐, 부품이 아님 |

**핵심**: 프롬프트를 "문자열이 아니라 객체"로 다뤄야 파이프에 끼울 수 있습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human",  "{topic}의 장점 3가지를 각각 한 문장으로 알려줘."),
    ]
)

# ── 1. 값을 채워서 실제 메시지 생성 ────────────
messages = prompt.invoke({"level": "초보자", "topic": "파이썬"})
print("── prompt.invoke() 결과 ──────────────────")
print(messages)
print()

# 결과가 메시지 리스트다.
# 2교시(실습 4)에서 손으로 만들던 것이 자동 생성됐다.
for m in messages.to_messages():
    print(f"  {type(m).__name__:15s} {m.content}")

# ★ 중요한 관찰
#   prompt.invoke() 와 llm.invoke() 가 "같은 이름"이다.
#   우연이 아니라 설계다. 실습 5에서 그 이유가 드러난다.

In [ ]:
# ── 2. 템플릿은 무엇이 필요한지 스스로 안다 ────
print("필요한 변수:", prompt.input_variables)      # ['level', 'topic']
print()

# ── 3. 변수를 빠뜨리면? ────────────────────────
print("── topic 을 빠뜨리고 호출해 보면 ─────────")
try:
    prompt.invoke({"level": "초보자"})
except KeyError as e:
    first_line = str(e).strip('"').split("\\n")[0]
    print(f"  KeyError: {first_line}")
    print("  → 무엇이 빠졌는지 이름까지 알려준다. 명확한 에러로 즉시 발견.")

# f-string 이라면 NameError 가 나거나,
# 더 나쁘게는 엉뚱한 변수가 들어가 "조용히 잘못 동작"한다.

# ── 4. 같은 템플릿을 값만 바꿔 재사용 ──────────
print()
print("── 같은 템플릿, 다른 값 ──────────────────")
for values in [
    {"level": "초보자", "topic": "파이썬"},
    {"level": "실무자", "topic": "Git"},
]:
    human = prompt.invoke(values).to_messages()[1]
    print(f"  {values} → {human.content}")

## 3교시 2절 — `StrOutputParser`: 출력을 다듬는 부품

2교시에서 매번 이렇게 썼습니다.

```python
response = llm.invoke(...)
text = response.content      # ← 항상 이 한 줄이 붙는다
```

체인으로 연결하려면 이 변환도 **부품**이어야 합니다.

| 파서 | 역할 | 다루는 주차 |
|---|---|---|
| `StrOutputParser` | `AIMessage` → 순수 문자열 | **3주차 (지금)** |
| JSON / Pydantic 파서 | 텍스트 → 구조화된 객체 | 5주차 |

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(model=os.environ["MODEL"])
parser = StrOutputParser()

# 1) 모델 호출 → AIMessage
response = llm.invoke("파이썬을 한 문장으로 소개해줘.")
print("모델 반환 타입:", type(response))          # AIMessage

# 2) 파서에 넣으면 → str
result = parser.invoke(response)
print("파서 반환 타입:", type(result))            # <class 'str'>
print()
print("결과:", result.strip())

### 세 부품의 공통점

```
ChatPromptTemplate   .invoke(dict)        → messages
ChatOllama           .invoke(messages)    → AIMessage
StrOutputParser      .invoke(AIMessage)   → str
```

**발견**: 셋 다 `.invoke()` 를 가지고 있고, 앞 부품의 출력이 그대로 뒤 부품의 입력입니다.

→ 그렇다면 이어 붙일 수 있지 않을까?

## 실습 5 ★ (3교시) — 첫 번째 체인 `prompt | llm | parser`

**이것이 과제 1의 제출물입니다.** (저장소 경로: `week03/first_chain.py`)

```
    입력 dict
       │
       ▼
   ┌─────────┐  messages   ┌──────┐  AIMessage  ┌────────┐   str
   │ prompt  │ ──────────▶ │ llm  │ ──────────▶ │ parser │ ─────▶ 출력
   └─────────┘             └──────┘             └────────┘
       └─────────────────── chain ───────────────────┘
```

이 짧은 한 줄이 5주차 LCEL, 10~11주차 RAG 체인, 12주차 그래프의 기본형입니다.

> ⚠️ **제출은 `.py` 파일로 합니다.** 이 노트북에서 동작을 확인한 뒤,
> 아래 셀들을 합쳐 `week03/first_chain.py` 로 저장해 커밋하십시오.
> (마지막 셀에 저장용 코드를 넣어 두었습니다)

In [ ]:
import os, time
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

MODEL = os.environ["MODEL"]

# ── 부품 3개 ──────────────────────────────────────
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human",  "{topic}의 장점 3가지를 각각 한 문장으로 알려줘."),
    ]
)
llm = ChatOllama(model=MODEL)
parser = StrOutputParser()

print("부품 3개 준비 완료")

In [ ]:
# ── 방법 A: 손으로 하나씩 ─────────────────
#
# 동작은 한다. 하지만 중간 변수가 3개 필요하고,
# 부품이 늘어날수록 코드가 계속 길어진다.

messages = prompt.invoke({"level": "초보자", "topic": "파이썬"})
ai_msg   = llm.invoke(messages)
text     = parser.invoke(ai_msg)

print(text.strip())

In [ ]:
# ── 방법 B: 파이프로 한 줄 ────────────────  ★
#
# | 는 파이썬의 __or__ 연산자를 LangChain 이 재정의한 것이다.
# 유닉스 파이프(cat file | grep x | wc -l)와 같은 발상.
#
# 이 구조에 이름이 있다 — LCEL (LangChain Expression Language).
# 5주차에서 본격적으로 다룬다.

chain = prompt | llm | parser          # ← 세 줄이 한 줄로

text = chain.invoke({"level": "초보자", "topic": "파이썬"})
print(text.strip())
print()

# 체인 자체도 .invoke() 를 가진다 → 다른 체인의 부품이 될 수 있다
print("체인의 타입:", type(chain))      # RunnableSequence

In [ ]:
# ── batch(): 여러 입력을 한 번에 ──────────
#
# 5주차 Self-Consistency(같은 질문을 여러 번 돌려 다수결)의 기반이 된다.

results = chain.batch(
    [
        {"level": "초보자", "topic": "파이썬"},
        {"level": "실무자", "topic": "Git"},
    ]
)

for r in results:
    print(r.strip()[:60], "...")
    print("-" * 40)

In [ ]:
# ── stream(): 토큰 단위 출력 ──────────────
#
# 답이 다 나올 때까지 기다리지 않고 글자가 흘러나온다.
# ChatGPT 의 그 느낌. 4주차에서 자세히 다룬다.

for chunk in chain.stream({"level": "초보자", "topic": "파이썬"}):
    print(chunk, end="", flush=True)
print()

### 정리

**실행 방식 3종**

| 메서드 | 무엇을 하나 | 어디서 다루나 |
|---|---|---|
| `invoke()` | 하나 실행 | 기본 |
| `batch()` | 여러 개 병렬 실행 | 5주차 Self-Consistency |
| `stream()` | 토큰 단위 스트리밍 | 4주차 |

10주차 RAG 에서는 앞에 부품이 하나 더 붙습니다:

```
retriever | prompt | llm | parser
```

## 제출물 만들기 — `week03/first_chain.py`

과제 1은 **`.py` 파일**로 제출합니다. 아래 셀을 실행하면 제출용 파일이 생성되고,
좌측 📁 파일 패널에서 내려받을 수 있습니다.

In [ ]:
CODE = '''"""[3교시 / 실습 5] 첫 번째 체인 — prompt | llm | parser

과제 1 제출물 (저장소 경로: week03/first_chain.py)
"""

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

MODEL = "gemma3:4b"          # 실습실 모델이 다르면 이 한 줄만 고친다

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human",  "{topic}의 장점 3가지를 각각 한 문장으로 알려줘."),
    ]
)
llm = ChatOllama(model=MODEL)
parser = StrOutputParser()

chain = prompt | llm | parser


def main() -> None:
    print(chain.invoke({"level": "초보자", "topic": "파이썬"}))


if __name__ == "__main__":
    main()
'''

import pathlib
pathlib.Path("first_chain.py").write_text(CODE, encoding="utf-8")
print("✅ first_chain.py 생성 완료 — 좌측 📁 파일 패널에서 내려받으세요")
print("   저장소에는 week03/first_chain.py 경로로 커밋합니다.")

## 오늘 확인할 것

- [ ] `ChatOllama.invoke()` 의 반환이 **객체**임을 확인했다
- [ ] `ollama.chat()` 과 `ChatOllama` 의 **결과 접근 방식** 차이를 말할 수 있다
- [ ] `SystemMessage` / `HumanMessage` / `AIMessage` 를 구분해 쓸 수 있다
- [ ] `prompt | llm | parser` 를 직접 조립했다 ★
- [ ] `first_chain.py` 를 저장소에 커밋했다 (과제 1)

> **실습실에서 따로 할 것**: `python -m venv .venv` · VS Code 인터프리터 지정 ·
> `.gitignore` 에 `.env` 등록 · `git check-ignore -v .env` 확인